# Comprehensive Lahman Baseball Database Builder

The original `project.ipynb` loads only **5 of 27** available Lahman CSV files (People, Batting, Pitching, Teams, Salaries). This notebook builds a complete SQLite database from **all 27 tables** with proper primary keys, foreign keys, and referential integrity.

### Tables loaded

| Category | Tables |
|----------|--------|
| **Core** | People, Teams, TeamsFranchises, TeamsHalf |
| **Batting** | Batting, BattingPost |
| **Pitching** | Pitching, PitchingPost |
| **Fielding** | Fielding, FieldingOF, FieldingOFsplit, FieldingPost |
| **Awards** | AwardsPlayers, AwardsManagers, AwardsSharePlayers, AwardsShareManagers |
| **Other** | Salaries, AllstarFull, Appearances, HallOfFame, Managers, ManagersHalf |
| **Reference** | Parks, Schools, CollegePlaying, HomeGames, SeriesPost |

## 1. Setup

In [ ]:
import pandas as pd
import sqlite3
import os

DATA_DIR = "data"
DB_PATH = "lahman_full.db"

# Remove existing database to start fresh
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print(f"Removed existing {DB_PATH}")

# List available CSVs
csv_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".csv")])
print(f"Found {len(csv_files)} CSV files:\n")
for f in csv_files:
    print(f"  {f}")

## 2. Load all CSVs into DataFrames

In [ ]:
# Load all CSVs into a dict of DataFrames
# Table name is derived from the CSV filename (without extension)
dataframes = {}

for f in csv_files:
    table_name = f.replace(".csv", "")
    df = pd.read_csv(os.path.join(DATA_DIR, f), encoding="utf-8-sig", low_memory=False)
    dataframes[table_name] = df
    print(f"{table_name:25s} {len(df):>7,} rows x {len(df.columns):>2} cols")

print(f"\nTotal: {sum(len(df) for df in dataframes.values()):,} rows across {len(dataframes)} tables")

## 3. Define Schema — Primary Keys and Foreign Keys

Every table gets a primary key. Foreign keys enforce referential integrity across the database. Tables are loaded in dependency order (parent tables first).

In [ ]:
# Schema definition: primary keys and foreign keys for all tables
# FK format: (local_cols, ref_table, ref_cols) — cols are comma-separated for composites

schema = {
    # --- Reference / Lookup tables (no foreign keys) ---
    "TeamsFranchises": {
        "pk": ["franchID"],
        "fk": [],
    },
    "Parks": {
        "pk": ["parkkey"],
        "fk": [],
    },
    "Schools": {
        "pk": ["schoolID"],
        "fk": [],
    },
    "People": {
        "pk": ["playerID"],
        "fk": [],
    },

    # --- Core tables ---
    "Teams": {
        "pk": ["yearID", "teamID"],
        "fk": [
            ("franchID", "TeamsFranchises", "franchID"),
        ],
    },
    "TeamsHalf": {
        "pk": ["yearID", "teamID", "Half"],
        "fk": [
            ("yearID,teamID", "Teams", "yearID,teamID"),
        ],
    },

    # --- Batting ---
    "Batting": {
        "pk": ["playerID", "yearID", "stint"],
        "fk": [
            ("playerID", "People", "playerID"),
            ("yearID,teamID", "Teams", "yearID,teamID"),
        ],
    },
    "BattingPost": {
        "pk": ["playerID", "yearID", "round"],
        "fk": [
            ("playerID", "People", "playerID"),
            ("yearID,teamID", "Teams", "yearID,teamID"),
        ],
    },

    # --- Pitching ---
    "Pitching": {
        "pk": ["playerID", "yearID", "stint"],
        "fk": [
            ("playerID", "People", "playerID"),
            ("yearID,teamID", "Teams", "yearID,teamID"),
        ],
    },
    "PitchingPost": {
        "pk": ["playerID", "yearID", "round"],
        "fk": [
            ("playerID", "People", "playerID"),
            ("yearID,teamID", "Teams", "yearID,teamID"),
        ],
    },

    # --- Fielding ---
    "Fielding": {
        "pk": ["playerID", "yearID", "stint", "POS"],
        "fk": [
            ("playerID", "People", "playerID"),
            ("yearID,teamID", "Teams", "yearID,teamID"),
        ],
    },
    "FieldingOF": {
        "pk": ["playerID", "yearID", "stint"],
        "fk": [
            ("playerID", "People", "playerID"),
        ],
    },
    "FieldingOFsplit": {
        "pk": ["playerID", "yearID", "stint", "POS"],
        "fk": [
            ("playerID", "People", "playerID"),
            ("yearID,teamID", "Teams", "yearID,teamID"),
        ],
    },
    "FieldingPost": {
        "pk": ["playerID", "yearID", "round", "POS"],
        "fk": [
            ("playerID", "People", "playerID"),
            ("yearID,teamID", "Teams", "yearID,teamID"),
        ],
    },

    # --- Awards ---
    "AwardsPlayers": {
        "pk": ["playerID", "awardID", "yearID", "lgID"],
        "fk": [
            ("playerID", "People", "playerID"),
        ],
    },
    "AwardsManagers": {
        "pk": ["playerID", "awardID", "yearID", "lgID"],
        "fk": [
            ("playerID", "People", "playerID"),
        ],
    },
    "AwardsSharePlayers": {
        "pk": ["awardID", "yearID", "lgID", "playerID"],
        "fk": [
            ("playerID", "People", "playerID"),
        ],
    },
    "AwardsShareManagers": {
        "pk": ["awardID", "yearID", "lgID", "playerID"],
        "fk": [
            ("playerID", "People", "playerID"),
        ],
    },

    # --- Other player tables ---
    "Salaries": {
        "pk": ["yearID", "teamID", "playerID"],
        "fk": [
            ("playerID", "People", "playerID"),
            ("yearID,teamID", "Teams", "yearID,teamID"),
        ],
    },
    "AllstarFull": {
        "pk": ["playerID", "yearID", "gameNum"],
        "fk": [
            ("playerID", "People", "playerID"),
        ],
    },
    "Appearances": {
        "pk": ["yearID", "teamID", "playerID"],
        "fk": [
            ("playerID", "People", "playerID"),
            ("yearID,teamID", "Teams", "yearID,teamID"),
        ],
    },
    "HallOfFame": {
        "pk": ["playerID", "yearid", "votedBy"],
        "fk": [
            ("playerID", "People", "playerID"),
        ],
    },
    "Managers": {
        "pk": ["playerID", "yearID", "inseason"],
        "fk": [
            ("playerID", "People", "playerID"),
            ("yearID,teamID", "Teams", "yearID,teamID"),
        ],
    },
    "ManagersHalf": {
        "pk": ["playerID", "yearID", "teamID", "half"],
        "fk": [
            ("playerID", "People", "playerID"),
            ("yearID,teamID", "Teams", "yearID,teamID"),
        ],
    },
    "CollegePlaying": {
        "pk": ["playerID", "schoolID", "yearID"],
        "fk": [
            ("playerID", "People", "playerID"),
            ("schoolID", "Schools", "schoolID"),
        ],
    },
    "HomeGames": {
        "pk": ["yearkey", "teamkey", "parkkey"],
        "fk": [
            ("parkkey", "Parks", "parkkey"),
        ],
    },
    "SeriesPost": {
        "pk": ["yearID", "round"],
        "fk": [],
    },
}

print(f"Schema defined for {len(schema)} tables")
print(f"Tables in data dir: {len(dataframes)}")
assert set(schema.keys()) == set(dataframes.keys()), f"Mismatch: {set(schema.keys()) ^ set(dataframes.keys())}"

## 4. Handle duplicate primary keys

Some Lahman CSVs contain duplicate rows for the defined primary keys. We deduplicate before inserting so primary key constraints hold.

In [ ]:
# Check for and remove duplicate primary keys
for table_name, config in schema.items():
    df = dataframes[table_name]
    pk = config["pk"]
    dupes = df.duplicated(subset=pk, keep="first")
    n_dupes = dupes.sum()
    if n_dupes > 0:
        print(f"{table_name}: dropping {n_dupes} duplicate rows (by {pk})")
        dataframes[table_name] = df.drop_duplicates(subset=pk, keep="first")
    
print("\nDeduplication complete.")

## 5. Create tables and load data

Tables are created in topological order (parents before children) so foreign key constraints can be validated. The `ID` column present in some CSVs (People, Parks, Schools, Teams) is dropped since it's just a row number.

In [ ]:
# Topological load order: parents first, then children
load_order = [
    # Tier 0: no dependencies
    "TeamsFranchises", "Parks", "Schools", "People",
    # Tier 1: depends on tier 0
    "Teams", "SeriesPost",
    # Tier 2: depends on tier 1
    "TeamsHalf", "Batting", "BattingPost", "Pitching", "PitchingPost",
    "Fielding", "FieldingOF", "FieldingOFsplit", "FieldingPost",
    "Salaries", "AllstarFull", "Appearances", "HallOfFame",
    "Managers", "ManagersHalf", "CollegePlaying", "HomeGames",
    "AwardsPlayers", "AwardsManagers", "AwardsSharePlayers", "AwardsShareManagers",
]

assert set(load_order) == set(schema.keys()), "Load order doesn't match schema"

# Map pandas dtypes to SQLite types
def sql_type(dtype):
    s = str(dtype)
    if "int" in s:
        return "INTEGER"
    elif "float" in s:
        return "REAL"
    return "TEXT"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

for table_name in load_order:
    config = schema[table_name]
    df = dataframes[table_name].copy()
    
    # Drop the spurious 'ID' column (row number) if present
    if "ID" in df.columns and "ID" not in config["pk"]:
        df = df.drop(columns=["ID"])
    
    # Build CREATE TABLE statement
    col_defs = []
    for col in df.columns:
        col_defs.append(f'"{col}" {sql_type(df[col].dtype)}')
    
    parts = [f"CREATE TABLE \"{table_name}\" ("]
    parts.append("  " + ",\n  ".join(col_defs))
    
    # Composite primary key
    pk_cols = ", ".join(f'"{c}"' for c in config["pk"])
    parts.append(f"  , PRIMARY KEY ({pk_cols})")
    
    # Foreign keys
    for fk_cols_str, ref_table, ref_cols_str in config["fk"]:
        fk_cols = ", ".join(f'"{c.strip()}"' for c in fk_cols_str.split(","))
        ref_cols = ", ".join(f'"{c.strip()}"' for c in ref_cols_str.split(","))
        parts.append(f'  , FOREIGN KEY ({fk_cols}) REFERENCES "{ref_table}" ({ref_cols})')
    
    parts.append(");")
    create_sql = "\n".join(parts)
    
    cursor.execute(f'DROP TABLE IF EXISTS "{table_name}";')
    cursor.execute(create_sql)
    
    # Load data
    df.to_sql(table_name, conn, if_exists="append", index=False)
    
    # Verify row count
    count = cursor.execute(f'SELECT COUNT(*) FROM "{table_name}"').fetchone()[0]
    status = "OK" if count == len(df) else "MISMATCH"
    print(f"  {table_name:25s} {count:>7,} rows  [{status}]")

conn.commit()
print(f"\nDatabase created: {DB_PATH}")

## 6. Verify database integrity

In [ ]:
# Verify: list all tables, row counts, and foreign key integrity
print("=== Tables in database ===\n")
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)
for _, row in tables.iterrows():
    count = pd.read_sql_query(f'SELECT COUNT(*) as n FROM "{row["name"]}"', conn).iloc[0, 0]
    print(f"  {row['name']:25s} {count:>7,} rows")

print(f"\n  {'TOTAL':25s} {sum(pd.read_sql_query(f'SELECT COUNT(*) as n FROM \"{t}\"', conn).iloc[0,0] for t in tables['name']):>7,} rows")

# Run foreign key check
fk_check = pd.read_sql_query("PRAGMA foreign_key_check;", conn)
if len(fk_check) == 0:
    print("\nForeign key integrity: PASSED")
else:
    print(f"\nForeign key violations: {len(fk_check)}")
    print(fk_check.head(20))

# Run integrity check
integrity = cursor.execute("PRAGMA integrity_check;").fetchone()[0]
print(f"Integrity check: {integrity}")

## 7. Quick sanity queries

A few cross-table queries to confirm relationships work.

In [ ]:
def run_query(query, params=None):
    return pd.read_sql_query(query, conn, params=params)

# Career HR leaders with full names (People + Batting)
print("=== Career Home Run Leaders ===\n")
run_query("""
    SELECT p.nameFirst || ' ' || p.nameLast AS name,
           SUM(b.HR) AS career_HR,
           MIN(b.yearID) AS first_yr,
           MAX(b.yearID) AS last_yr
    FROM Batting b
    JOIN People p ON b.playerID = p.playerID
    GROUP BY b.playerID
    ORDER BY career_HR DESC
    LIMIT 10
""")

In [ ]:
# Gold Glove winners with fielding stats (AwardsPlayers + Fielding + People)
print("=== Gold Glove Winners — Most Awards ===\n")
run_query("""
    SELECT p.nameFirst || ' ' || p.nameLast AS name,
           COUNT(*) AS gold_gloves
    FROM AwardsPlayers a
    JOIN People p ON a.playerID = p.playerID
    WHERE a.awardID = 'Gold Glove'
    GROUP BY a.playerID
    ORDER BY gold_gloves DESC
    LIMIT 10
""")

In [ ]:
# Hall of Famers who played college ball (HallOfFame + CollegePlaying + Schools + People)
print("=== Hall of Famers with College Background ===\n")
run_query("""
    SELECT DISTINCT
           p.nameFirst || ' ' || p.nameLast AS name,
           s.name_full AS college,
           s.state
    FROM HallOfFame h
    JOIN People p ON h.playerID = p.playerID
    JOIN CollegePlaying c ON h.playerID = c.playerID
    JOIN Schools s ON c.schoolID = s.schoolID
    WHERE h.inducted = 'Y' AND h.category = 'Player'
    ORDER BY p.nameLast
    LIMIT 15
""")

In [ ]:
# Postseason batting leaders (BattingPost + People + SeriesPost)
print("=== All-Time Postseason HR Leaders ===\n")
run_query("""
    SELECT p.nameFirst || ' ' || p.nameLast AS name,
           SUM(bp.HR) AS postseason_HR,
           SUM(bp.H) AS postseason_H,
           COUNT(DISTINCT bp.yearID) AS seasons
    FROM BattingPost bp
    JOIN People p ON bp.playerID = p.playerID
    GROUP BY bp.playerID
    HAVING postseason_HR > 0
    ORDER BY postseason_HR DESC
    LIMIT 10
""")

In [ ]:
# Top ballparks by total attendance (HomeGames + Parks)
print("=== Ballparks — Highest Total Attendance ===\n")
run_query("""
    SELECT pk.parkname,
           pk.city || ', ' || pk.state AS location,
           SUM(hg.attendance) AS total_attendance,
           MIN(hg.yearkey) AS first_yr,
           MAX(hg.yearkey) AS last_yr
    FROM HomeGames hg
    JOIN Parks pk ON hg.parkkey = pk.parkkey
    WHERE hg.attendance IS NOT NULL
    GROUP BY hg.parkkey
    ORDER BY total_attendance DESC
    LIMIT 10
""")

In [ ]:
conn.close()

db_size_mb = os.path.getsize(DB_PATH) / (1024 * 1024)
print(f"Done. Database saved to {DB_PATH} ({db_size_mb:.1f} MB)")